# Notebook 2: Building your own pipeline with QSARmil modules

`MultiConformerRegressor`/`MultiConformerClassifier` do everything for you automatically. 
This notebook opens up that black box: it walks through each step separately (generating
conformers, computing descriptors, training models, optimizing hyperparamters), explains exactly what data goes in and
comes out of each step, and shows every setting you can change along the way.

This is the notebook to read if you want to:
- plug in your own conformer generator
- plug in your own descriptor calculator
- plug in your own MIL method
- benchmark different conformer generators, descriptors, etc.
- extend QSARmil pipeline to more complex cases (e.g. for reaction property modelling)

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "" # uncomment if you do not have GPU

In [2]:
import numpy as np
import pandas as pd

### Load data

As an example, we use a small, publicly available dataset of binding activities, from:

> Van Tilborg, Derek, Alisa Alenicheva, and Francesca Grisoni. "Exposing the limitations of molecular machine
> learning with activity cliffs." Journal of chemical information and modeling 62.23 (2022): 5938-5951.

In [3]:
url = "https://raw.githubusercontent.com/molML/MoleculeACE/main/MoleculeACE/Data/benchmark_data/CHEMBL2034_Ki.csv"
df_ace = pd.read_csv(url)

df_train = df_ace[df_ace["split"] == "train"][["smiles", "y"]].reset_index(drop=True)
df_test = df_ace[df_ace["split"] == "test"][["smiles", "y"]].reset_index(drop=True)

df_train.shape, df_test.shape

((598, 2), (152, 2))

<div class="alert alert-block alert-info">
<b>💡 Tip: </b>Training a full model on the whole dataset can take a while, since QSARmil tries many different model
combinations. If you just want to try things out quickly (for example, to check that everything runs on your
machine), keep the cell below uncommented to work with a small random sample instead. Comment it out once
you're ready to run the real thing.
</div>

In [4]:
df_train = df_train.sample(n=15, random_state=42).reset_index(drop=True)
df_test = df_test.sample(n=5, random_state=42).reset_index(drop=True)

df_train.shape, df_test.shape

((15, 2), (5, 2))

### Step-1. Parse SMILES

In [5]:
from rdkit import Chem

In [6]:
mols_train = [Chem.MolFromSmiles(smi) for smi in df_train["smiles"]]
mols_test = [Chem.MolFromSmiles(smi) for smi in df_test["smiles"]]

print(f"Number of training molecules: {len(mols_train)}")
print(f"Number of test molecules: {len(mols_test)}")

Number of training molecules: 15
Number of test molecules: 5


### Step-2. Conformer generation

`QSARmil` currently uses a minimalistic class (`RDKitConformerGenerator`) wrapping the RDKit conformer generation pipeline. For each molecule, conformers are generated, then each molecule is split into individual conformers (attached to the molecule copy).

**Important:**
If conformer generation failed, it will return a `FailedMolecule` object. For model training, they have to be removed, for test molecules, predictions for them cannot be generated and should be replaced by a baseline prediction.

`RDKitConformerGenerator` accepts parameters:

- `num_conf` - maximum number of conformers to generate per molecule
- `e_thresh` - energy cutoff, any conformer more than the most stable one is removed. `None` disables this filtering. Default: `None`.
- `num_cpu` - number of CPU threads to use. Default: `1`
- `verbose` - whether to print a progress counter
- `random_seed` - random seed for conformer embedding, for reproducible results

If you want to use your **own** conformer generator, it just needs to look like the above: something that returns a list of plain molecules with one conformer per molecule copy, because this format is expected by the descriptor calculator.

In [7]:
from qsarmil.conformer import RDKitConformerGenerator

In [8]:
conf_gen = RDKitConformerGenerator(num_conf=5, e_thresh=50, num_cpu=2, verbose=True, random_seed=42)

In [9]:
confs_train = conf_gen.run(mols_train)

Generating conformers: 15/15


In [10]:
confs_test = conf_gen.run(mols_test)

Generating conformers: 5/5


In [11]:
confs_train[:3]

[[<rdkit.Chem.rdchem.Mol at 0x7f97c7dd7510>,

### 3. Descriptor calculation

QSARmil's `DescriptorWrapper` handles this for you: it takes any descriptor calculator (from RDKit, from MolFeat,
or your own) and applies it to every conformer of every molecule.


QSARmil ships several ready-to-use 3D descriptor types:
- **RDKit-based:** `RDKitGEOM`, `RDKitAUTOCORR`, `RDKitRDF`, `RDKitMORSE`, `RDKitWHIM`, `RDKitGETAWAY`
- **MolFeat-based:** `Pharmacophore3D`, `USRDescriptors`, `ElectroShapeDescriptors`

To use your **own** descriptor, all it needs to be is a callable that takes one conformer (a single-conformer
`Mol`) and returns a 1D NumPy array of numbers, and it works exactly like the built-in ones below.

In [12]:
from qsarmil.descriptor.wrapper import DescriptorWrapper
from molfeat.calc import Pharmacophore3D
from milearn.preprocessing import BagMinMaxScaler

In [13]:
desc_calc = DescriptorWrapper(Pharmacophore3D(factory="pmapper"), verbose=True)

In [14]:
x_train = desc_calc.run(confs_train)

Calculating descriptors: 15/15

In [15]:
x_test = desc_calc.run(confs_test)

Calculating descriptors: 5/5

One last step before training: descriptor values are usually on very different scales (some might range in the
thousands, others between -1 and 1), which makes training harder. `BagMinMaxScaler` rescales every column to the
same `[0, 1]` range - fit it on the training data only, then reuse that same fit to transform both splits.

In [16]:
scaler = BagMinMaxScaler()
scaler.fit(x_train)

x_train_scaled = scaler.transform(x_train)
x_test_scaled = scaler.transform(x_test)

y_train, y_test = df_train["y"], df_test["y"]

### 4. Multi-instance method application

Here, each molecule is a *bag* of several conformers encoded with 3D descriptors instead. QSARmil (via the `milearn` package) offers a few different families to train a model on bags. Below, copy and paste any of them to import.

Every one of these accepts the same core settings (they all build on the same underlying `BaseNetwork`):

**Basic neural network parameters:**

- `hidden_layer_sizes` - sizes of the hidden layers in the network, e.g., `(256, 128, 64)`
- `max_epochs` - maximum number of passes over the training data
- `batch_size` - number of molecules processed together in one training step
- `activation` - the activation function used between layers (e.g. `"relu"`, `"gelu"`, `"elu"`, `"silu"`, `"leakyrelu"`)
- `learning_rate` - how big a step the optimizer takes at each update. Smaller is more cautious but slower. Default: `0.001`
- `early_stopping` - whether to stop training automatically once the model stops improving, instead of always. Default: `True`
- `weight_decay` - a regularization strength that discourages overly large weights (helps avoid overfitting). Default: `0.0`
- `accelerator` - `"cpu"` or `"gpu"` - which hardware to train on. Default: `"cpu"`
- `verbose` - whether to print training progress. Default: `False`
- `random_seed` - fixed random seed for reproducible training

**Multi-instance special parameters:**

- `instance_dropout` - fraction of instances randomly ignored during each training step, as a way to avoid overfitting. Default: `0.0`.
- `pool` (on `BagNetwork`/`InstanceNetwork`/the wrapper networks) - how conformers are combined: `"mean"`, `"sum"`, `"max"`, or `"lse"`.
- `tau` (on the attention-based networks) - a temperature that controls how "sharp" or "soft" the learned attention weights are

You can apply any other MIL method that can be trained on bags.

In [17]:
from milearn.network.regressor import (
                                       BagNetworkRegressor,
                                       InstanceNetworkRegressor,
                                       AdditiveAttentionNetworkRegressor,
                                       SelfAttentionNetworkRegressor,
                                       HopfieldAttentionNetworkRegressor,
                                       DynamicPoolingNetworkRegressor,
                                      )

In [18]:
model = AdditiveAttentionNetworkRegressor(
    hidden_layer_sizes=(256, 128, 64),  # 3 hidden layers of these sizes
    max_epochs=1000,                      # small, so this cell runs quickly on the sample data
    batch_size=128,
    activation="gelu",
    learning_rate=0.001,
    early_stopping=True,
    weight_decay=0.0,
    instance_dropout=0.0,
    accelerator="cpu",
    verbose=False,
    random_seed=42,
    tau=1.0,                             # extra setting specific to attention-based networks
)
model.fit(x_train_scaled, y_train)
y_pred = model.predict(x_test_scaled)
y_pred

array([-0.21120304, -1.5839748 , -0.44138464, -0.43012124, -0.41925883],
      dtype=float32)

### 5. Automatic hyperparameter search (optional)

Instead of choosing every setting above by hand, you can let `milearn` search for good values itself, one setting
at a time. This can take a while (it trains many candidate models).

In [19]:
from milearn.network.module.hopt import DEFAULT_PARAM_GRID

In [20]:
DEFAULT_PARAM_GRID

{'max_epochs': 1000,
 'early_stopping': True,
 'accelerator': 'cpu',
 'random_seed': 42,
 'verbose': False,
 'activation': ['relu', 'leakyrelu', 'gelu', 'elu', 'silu'],
 'learning_rate': [0.0001, 0.001],
 'batch_size': [32, 512, 1024],
 'weight_decay': [0.0, 1e-05, 0.0001, 0.001, 0.01],
 'tau': [0.01, 0.5, 1.0],
 'instance_dropout': [0.0, 0.2, 0.4, 0.6, 0.8],
 'pool': ['mean', 'sum', 'max', 'lse'],
 'hidden_layer_sizes': [(2048, 1024, 512, 256, 128, 64),
  (256, 128, 64),
  (128,)]}

In [21]:
model = AdditiveAttentionNetworkRegressor()
model.hopt(x_train_scaled, y_train, param_grid=DEFAULT_PARAM_GRID, verbose=True)
model.fit(x_train_scaled, y_train)
y_pred = model.predict(x_test_scaled)
y_pred

Optimizing hyperparameter: activation (5 options)
[1/26 |  3.8% |  0.0 min] Value: relu, Epochs: 18, Loss: 1.8947
[2/26 |  7.7% |  0.0 min] Value: leakyrelu, Epochs: 22, Loss: 1.7701
[3/26 | 11.5% |  0.0 min] Value: gelu, Epochs: 13, Loss: 2.4168
[4/26 | 15.4% |  0.0 min] Value: elu, Epochs: 25, Loss: 0.8252
[5/26 | 19.2% |  0.0 min] Value: silu, Epochs: 20, Loss: 1.3163
Best activation = elu, val_loss = 0.8252
Optimizing hyperparameter: learning_rate (2 options)
[6/26 | 23.1% |  0.0 min] Value: 0.0001, Epochs: 19, Loss: 2.5028
[7/26 | 26.9% |  0.0 min] Value: 0.001, Epochs: 30, Loss: 0.7933
Best learning_rate = 0.001, val_loss = 0.7933
Optimizing hyperparameter: batch_size (3 options)
[8/26 | 30.8% |  0.0 min] Value: 32, Epochs: 29, Loss: 1.0102
[9/26 | 34.6% |  0.0 min] Value: 512, Epochs: 38, Loss: 2.6502
[10/26 | 38.5% |  0.0 min] Value: 1024, Epochs: 29, Loss: 1.0580
Best batch_size = 32, val_loss = 1.0102
Optimizing hyperparameter: weight_decay (5 options)
[11/26 | 42.3% |  0.1 m

array([-1.2383875, -2.6213956, -2.0208015, -2.4664333, -1.8020129],
      dtype=float32)